# Calibração Conforme dos Grupos A/B/C (S5.7) — Kaggle T4

**Autor**: Massanori  
**Data**: 21/05/2026  
**Descrição**: Orquestra a calibração conforme dos 3 grupos do TCC sobre o split `cal`. Para cada grupo, carrega o `best.pt` do dataset Kaggle correspondente, computa o score de inconformidade pixelwise, e salva o `q_hat` em JSON com metadata + SHA-256 do checkpoint (para auditoria da banca).

## Pré-requisitos

- Settings → Accelerator = **T4 x1**
- Add Input (5 datasets):
  1. `tcc-mri-recons-varnet-brain-4x` — split `cal` (S4)
  2. `tcc-mri-resm-checkpoints` — best.pt do Grupo A
  3. `tcc-mri-qr-checkpoints` — best.pt do Grupo B
  4. `tcc-mri-qr-lesion-checkpoints` — best.pt do Grupo C
  5. (Opcional) `tcc-mri-lesion-masks` — não afeta `q_hat` em si, anexa por consistência

## Fluxo

1. Célula 2: clone do repo + dependencias
2. Célula 3: **diagnóstico** — valida GPU + localiza os 4 datasets esperados
3. Células 4–6: calibra Grupo A, B e C (ì~3 min cada em T4)
4. Célula 7: tabela comparativa dos 3 `q_hat`
5. Célula 8: publica `tcc-mri-conformal-qhats` (3 JSONs juntos)
6. Célula 9: markdown com plano S5.8

## Referência teórica

O `q_hat` calibrado garante cobertura marginal pixelwise sob exchangeability:

$$\mathbb{P}(y_{\text{pixel}} \in [\hat{l}(x), \hat{u}(x)] + q_\alpha) \geq 1 - \alpha$$

onde $q_\alpha$ é o quantil $(1-\alpha)(n+1)/n$-empírico dos scores de inconformidade. **Refs**: Romano, Patterson & Candès (2019, Teorema 1); Lei et al. (2018); Angelopoulos & Bates (2023). Para o paper-base, Giannakopoulos et al. (2026, Tabela 1) reporta `lambda_cal ~ 1.54` (escala equivalente do `q_hat`). Pearson check do Grupo B saiu em 0.978 — nossa replicação deve dar `q_hat` menor que 1.54.

In [10]:
# Célula 2 — Setup: clone repo + dependencias
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/KR0N0S7/tcc-mri-uncertainty.git'
REPO_DIR = Path('/kaggle/working/tcc-mri-uncertainty')

if REPO_DIR.is_dir():
    print(f'Repo ja existe em {REPO_DIR}; git pull...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('\nHEAD do repo:')
subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=False)

print('\nInstalando dependencias...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(REPO_DIR / 'requirements.txt'),
    'python-dotenv',
], check=True)
print('Setup OK')

Repo ja existe em /kaggle/working/tcc-mri-uncertainty; git pull...


From https://github.com/KR0N0S7/tcc-mri-uncertainty
   745e36e..765223f  main       -> origin/main


Updating 745e36e..765223f
Fast-forward
 scripts/calibrate.py | 99 +++++++++++++++-------------------------------------
 1 file changed, 28 insertions(+), 71 deletions(-)

HEAD do repo:
765223f fix: src import error.

Instalando dependencias...
Setup OK


In [11]:
# Célula 3 — Diagnóstico: GPU + localiza recons + 3 checkpoints
# Autor: Massanori
# Data: 21/05/2026
# Descrição: Diagnóstico de ambiente com fallback automático para CPU.
# Valida inputs Kaggle (recons + checkpoints), detecta dispositivo disponível
# e prepara variáveis globais para as células de calibração sem interromper
# execução quando não houver GPU.


RECONS_SLUG = 'tcc-mri-recons-varnet-brain-4x'
CKPT_SLUGS = {
    'A': 'tcc-mri-resm-checkpoints',
    'B': 'tcc-mri-qr-checkpoints',
    'C': 'tcc-mri-qr-lesion-checkpoints',
}
EXPECTED_RECONS_SUBS = {'train', 'val', 'cal', 'test'}
EXPECTED_CKPT_FILES = {'best.pt'}

def kaggle_input_candidates(slug):
    candidates = [Path('/kaggle/input') / slug]
    datasets_root = Path('/kaggle/input/datasets')
    if datasets_root.is_dir():
        for user_dir in datasets_root.iterdir():
            if user_dir.is_dir():
                candidates.append(user_dir / slug)
    return candidates

def find_recons_root(slug):
    for cand in kaggle_input_candidates(slug):
        if not cand.is_dir():
            continue
        subs = {p.name for p in cand.iterdir() if p.is_dir()}
        if EXPECTED_RECONS_SUBS.issubset(subs):
            return cand
        sub_dirs = [p for p in cand.iterdir() if p.is_dir()]
        if len(sub_dirs) == 1 and (sub_dirs[0] / 'train').is_dir():
            return sub_dirs[0]
    raise FileNotFoundError(
        f'Recons dataset "{slug}" não encontrado com train/val/cal/test.'
    )

def find_checkpoint_root(slug):
    """Encontra diretório com best.pt, tolerando múltiplos níveis/subpastas."""
    all_checked = []

    for cand in kaggle_input_candidates(slug):
        if not cand.is_dir():
            continue

        all_checked.append(str(cand))

        # Caso 1: best.pt direto na raiz
        if (cand / 'best.pt').is_file():
            return cand

        # Caso 2: busca recursiva por best.pt
        matches = [
            p for p in cand.rglob('best.pt')
            if '.ipynb_checkpoints' not in p.parts
        ]

        if matches:
            # Heurística para escolher a pasta mais "completa"
            ranked = []
            for m in matches:
                parent = m.parent
                score = 0
                if (parent / 'config_snapshot.json').is_file():
                    score += 2
                if (parent / 'metrics.csv').is_file():
                    score += 1
                if (parent / 'last.pt').is_file():
                    score += 1

                # Penaliza profundidade muito grande
                rel_depth = len(parent.relative_to(cand).parts)
                score -= rel_depth * 0.01

                ranked.append((score, parent))

            ranked.sort(key=lambda x: x[0], reverse=True)
            return ranked[0][1]

    raise FileNotFoundError(
        f'Checkpoint dataset "{slug}" não encontrado ou sem best.pt. '
        f'Caminhos verificados: {all_checked}'
    )

print('=' * 60)
print('DIAGNÓSTICO')
print('=' * 60)

print('\n[1] Conteúdo de /kaggle/input/:')
subprocess.run(['ls', '-la', '/kaggle/input/'], check=False)

import torch
HAS_CUDA = torch.cuda.is_available()
DEVICE = 'cuda' if HAS_CUDA else 'cpu'
print(f'\n[2] CUDA available: {HAS_CUDA}')
if HAS_CUDA:
    print(f'    GPU: {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'    VRAM: {vram_gb:.1f} GB')
else:
    print('    Executando em CPU (sem limite de GPU).')

print(f'\n[3] Recons dataset "{RECONS_SLUG}":')
RECONS_ROOT = find_recons_root(RECONS_SLUG)
print(f'  -> RECONS_ROOT: {RECONS_ROOT}')
n_cal = len(list((RECONS_ROOT / 'cal').glob('*.npz')))
print(f'  cal: {n_cal} arquivos .npz')
if n_cal == 0:
    raise FileNotFoundError(f'Split cal vazio em {RECONS_ROOT / "cal"}')

print(f'\n[4] Checkpoints:')
CHECKPOINT_ROOTS = {}
CHECKPOINT_PATHS = {}
for group, slug in CKPT_SLUGS.items():
    ckpt_root = find_checkpoint_root(slug)
    CHECKPOINT_ROOTS[group] = ckpt_root
    CHECKPOINT_PATHS[group] = ckpt_root / 'best.pt'
    size_mb = CHECKPOINT_PATHS[group].stat().st_size / 1e6
    print(f'  Grupo {group}: {ckpt_root}')
    print(f'    best.pt = {size_mb:.1f} MB')

os.environ['TCC_RECONS_DIR'] = str(RECONS_ROOT)
os.environ['TCC_ANOTADOS_DIR'] = '/kaggle/working/_dummy_anotados'
os.environ['TCC_BRAIN_CSV'] = '/kaggle/working/_dummy_brain.csv'
Path('/kaggle/working/_dummy_anotados').mkdir(exist_ok=True)
Path('/kaggle/working/_dummy_brain.csv').touch()

print('\n[5] Config do projeto:')
subprocess.run([sys.executable, '-m', 'src.config'], check=True)

print('\n' + '=' * 60)
print(f'DIAGNÓSTICO OK — prosseguir com DEVICE={DEVICE}')
print('=' * 60)

DIAGNÓSTICO

[1] Conteúdo de /kaggle/input/:
total 12
drwxr-xr-x 3 root root 4096 May 21 14:41 .
drwxr-xr-x 5 root root 4096 May 21 14:42 ..
drwxr-xr-x 3 root root 4096 May 21 14:41 datasets

[2] CUDA available: False
    Executando em CPU (sem limite de GPU).

[3] Recons dataset "tcc-mri-recons-varnet-brain-4x":
  -> RECONS_ROOT: /kaggle/input/datasets/massanorikishi/tcc-mri-recons-varnet-brain-4x
  cal: 46 arquivos .npz

[4] Checkpoints:
  Grupo A: /kaggle/input/datasets/massanorikishi/tcc-mri-resm-checkpoints/tcc-mri-resm-checkpoints
    best.pt = 93.1 MB
  Grupo B: /kaggle/input/datasets/massanorikishi/tcc-mri-qr-checkpoints
    best.pt = 186.2 MB
  Grupo C: /kaggle/input/datasets/massanorikishi/tcc-mri-qr-lesion-checkpoints
    best.pt = 186.2 MB

[5] Config do projeto:
Configuracao do projeto TCC:
  PROJECT_ROOT    = /kaggle/working/tcc-mri-uncertainty
  ANOTADOS_DIR    = /kaggle/working/_dummy_anotados
  BRAIN_CSV       = /kaggle/working/_dummy_brain.csv
  DATA_DIR        = /kag

In [12]:
# Célula 4 — Calibra Grupo A (ResM) via locally-adaptive scaled CP.
# Score: |y - x| / u(x). Intervalo calibrado: [x - q*u, x + q*u].
# Autor: Massanori
# Data: 21/05/2026
# Descrição: Calibra o Grupo A (ResM) usando DEVICE dinâmico definido na célula 3
# ('cuda' ou 'cpu'). Recebe checkpoint A e RECONS_ROOT; gera q_hat_A.json em
# /kaggle/working para auditoria e uso nas etapas S5.8/S5.9.


assert 'DEVICE' in globals(), "Variável DEVICE não encontrada. Execute a Célula 3 primeiro."
assert 'CHECKPOINT_PATHS' in globals(), "CHECKPOINT_PATHS não encontrado. Execute a Célula 3 primeiro."
assert 'RECONS_ROOT' in globals(), "RECONS_ROOT não encontrado. Execute a Célula 3 primeiro."

OUTPUT_A = '/kaggle/working/q_hat_A.json'

result = subprocess.run([
    sys.executable, 'scripts/calibrate.py',
    '--group', 'A',
    '--checkpoint', str(CHECKPOINT_PATHS['A']),
    '--recons-dir', str(RECONS_ROOT),
    '--output', OUTPUT_A,
    '--device', DEVICE,
    '--alpha', '0.10',
])

assert result.returncode == 0, f'Calibracao Grupo A falhou em {DEVICE}. Veja log acima.'
print(f'\nq_hat_A salvo em {OUTPUT_A} (device={DEVICE})')

2026-05-21 16:03:12,324 INFO calibrate: Device: cpu
2026-05-21 16:03:12,325 INFO calibrate: recons_root: /kaggle/input/datasets/massanorikishi/tcc-mri-recons-varnet-brain-4x
2026-05-21 16:03:12,325 INFO calibrate: checkpoint: /kaggle/input/datasets/massanorikishi/tcc-mri-resm-checkpoints/tcc-mri-resm-checkpoints/best.pt
2026-05-21 16:03:12,682 INFO calibrate: checkpoint SHA-256: 9ef2fa8e4e85706e2c90cdedc26cfe85dbf73987c6438b6070505072b493aeda
2026-05-21 16:03:12,757 INFO calibrate: Modulo Grupo A: 7,756,097 parametros
2026-05-21 16:03:12,858 INFO calibrate: Checkpoint carregado.
2026-05-21 16:03:15,428 INFO src.data.recons_dataset: ReconsSliceDataset: 730 slices em 46 volumes em cal
2026-05-21 16:03:15,428 INFO calibrate: Cal dataset: 730 slices
2026-05-21 16:09:01,515 WARNING calibrate: torch.quantile falhou por tensor grande; usando fallback numpy.partition.
2026-05-21 16:09:01,788 INFO src.calibration.conformal: Scaled CP calibration: n_batches=730, n_pixels=70,759,312, alpha=0.1, q


q_hat_A salvo em /kaggle/working/q_hat_A.json (device=cpu)


In [13]:
# Célula 5 — Calibra Grupo B (QR) via CQR (Romano et al., 2019).
# Score: max(lower - y, y - upper). Intervalo calibrado: [lower - q, upper + q].
# Autor: Massanori
# Data: 21/05/2026
# Descrição: Calibra o Grupo B (QR/CQR) usando DEVICE dinâmico definido na célula 3.
# Recebe checkpoint B e RECONS_ROOT; gera q_hat_B.json em /kaggle/working.


assert 'DEVICE' in globals(), "Variável DEVICE não encontrada. Execute a Célula 3 primeiro."
assert 'CHECKPOINT_PATHS' in globals(), "CHECKPOINT_PATHS não encontrado. Execute a Célula 3 primeiro."
assert 'RECONS_ROOT' in globals(), "RECONS_ROOT não encontrado. Execute a Célula 3 primeiro."

OUTPUT_B = '/kaggle/working/q_hat_B.json'

result = subprocess.run([
    sys.executable, 'scripts/calibrate.py',
    '--group', 'B',
    '--checkpoint', str(CHECKPOINT_PATHS['B']),
    '--recons-dir', str(RECONS_ROOT),
    '--output', OUTPUT_B,
    '--device', DEVICE,
    '--alpha', '0.10',
])

assert result.returncode == 0, f'Calibracao Grupo B falhou em {DEVICE}. Veja log acima.'
print(f'\nq_hat_B salvo em {OUTPUT_B} (device={DEVICE})')

2026-05-21 16:09:42,526 INFO calibrate: Device: cpu
2026-05-21 16:09:42,526 INFO calibrate: recons_root: /kaggle/input/datasets/massanorikishi/tcc-mri-recons-varnet-brain-4x
2026-05-21 16:09:42,526 INFO calibrate: checkpoint: /kaggle/input/datasets/massanorikishi/tcc-mri-qr-checkpoints/best.pt
2026-05-21 16:09:44,485 INFO calibrate: checkpoint SHA-256: fc185dedb3457b3cec41e7640af3d35e8d968d4ae5f9431a3267952da516080a
2026-05-21 16:09:44,630 INFO calibrate: Modulo Grupo B: 15,512,194 parametros
2026-05-21 16:09:44,832 INFO calibrate: Checkpoint carregado.
2026-05-21 16:09:47,286 INFO src.data.recons_dataset: ReconsSliceDataset: 730 slices em 46 volumes em cal
2026-05-21 16:09:47,286 INFO calibrate: Cal dataset: 730 slices
2026-05-21 16:20:32,533 WARNING calibrate: torch.quantile falhou por tensor grande; usando fallback numpy.partition.
2026-05-21 16:20:32,821 INFO src.calibration.conformal: CQR calibration: n_batches=730, n_pixels=70,759,312, alpha=0.1, q_hat=0.010275, mean_score=0.0017


q_hat_B salvo em /kaggle/working/q_hat_B.json (device=cpu)


In [14]:
# Célula 6 — Calibra Grupo C (QR-Lesion) via CQR.
# Mesmo metodo do B (a contribuicao do C atua na loss, nao no calibrador).
# Mascaras nao sao necessarias para a calibracao em si — entram so no S5.8.
# Autor: Massanori
# Data: 21/05/2026
# Descrição: Calibra o Grupo C (QR-Lesion/CQR) usando DEVICE dinâmico definido
# na célula 3. Recebe checkpoint C e RECONS_ROOT; gera q_hat_C.json em
# /kaggle/working para comparação final na célula 7.


assert 'DEVICE' in globals(), "Variável DEVICE não encontrada. Execute a Célula 3 primeiro."
assert 'CHECKPOINT_PATHS' in globals(), "CHECKPOINT_PATHS não encontrado. Execute a Célula 3 primeiro."
assert 'RECONS_ROOT' in globals(), "RECONS_ROOT não encontrado. Execute a Célula 3 primeiro."

OUTPUT_C = '/kaggle/working/q_hat_C.json'

result = subprocess.run([
    sys.executable, 'scripts/calibrate.py',
    '--group', 'C',
    '--checkpoint', str(CHECKPOINT_PATHS['C']),
    '--recons-dir', str(RECONS_ROOT),
    '--output', OUTPUT_C,
    '--device', DEVICE,
    '--alpha', '0.10',
])

assert result.returncode == 0, f'Calibracao Grupo C falhou em {DEVICE}. Veja log acima.'
print(f'\nq_hat_C salvo em {OUTPUT_C} (device={DEVICE})')

2026-05-21 16:22:09,483 INFO calibrate: Device: cpu
2026-05-21 16:22:09,483 INFO calibrate: recons_root: /kaggle/input/datasets/massanorikishi/tcc-mri-recons-varnet-brain-4x
2026-05-21 16:22:09,483 INFO calibrate: checkpoint: /kaggle/input/datasets/massanorikishi/tcc-mri-qr-lesion-checkpoints/best.pt
2026-05-21 16:22:11,167 INFO calibrate: checkpoint SHA-256: fa5198f832acd58fc8f0ed00a244dd92a86a8e3d2c7b7a8e57d0b20bc8cc58fe
2026-05-21 16:22:11,311 INFO calibrate: Modulo Grupo C: 15,512,194 parametros
2026-05-21 16:22:11,509 INFO calibrate: Checkpoint carregado.
2026-05-21 16:22:13,872 INFO src.data.recons_dataset: ReconsSliceDataset: 730 slices em 46 volumes em cal
2026-05-21 16:22:13,872 INFO calibrate: Cal dataset: 730 slices
2026-05-21 16:33:09,644 WARNING calibrate: torch.quantile falhou por tensor grande; usando fallback numpy.partition.
2026-05-21 16:33:09,912 INFO src.calibration.conformal: CQR calibration: n_batches=730, n_pixels=70,759,312, alpha=0.1, q_hat=0.010269, mean_score


q_hat_C salvo em /kaggle/working/q_hat_C.json (device=cpu)


In [15]:
# Célula 7 — Tabela comparativa dos 3 q_hat
import json

results = {}
for group in 'ABC':
    p = Path(f'/kaggle/working/q_hat_{group}.json')
    results[group] = json.loads(p.read_text())

print('=' * 70)
print(f'CALIBRACAO CONCLUIDA \u2014 Grupos A/B/C (alpha=0.10, cobertura nominal 90%)')
print('=' * 70)
print(f'{"Grupo":<6}{"Metodo":<12}{"q_hat":<14}{"n_pixels":<16}{"mean_score":<12}')
print('-' * 70)
for group, r in results.items():
    print(
        f'{group:<6}'
        f'{r["method"]:<12}'
        f'{r["q_hat"]:<14.6f}'
        f'{r["n_pixels"]:<16,}'
        f'{r["mean_score"]:<12.6f}'
    )
print('=' * 70)

# Heads-up de leitura
print('\nReferencia (paper-base, Giannakopoulos et al. 2026, Tabela 1):')
print('  Grupo B (QR), lambda_cal esperado: ~1.54')
print('  Pearson check do Grupo B saiu em 0.978 (vs paper 0.91).')
print('  Predicao: q_hat_B < 1.54 (replicacao com calibracao melhor).')
print('  Predicao: q_hat_C ligeiramente menor que q_hat_B (C ja predisse')
print('             intervalos um pouco mais largos, mean_unc 1.7% acima do B).')
print('  q_hat_A em escala diferente (multiplicativo: u_cal = q*u(x)),')
print('  esperado em [1.5, 2.0] para scaled CP locally-adaptive.')

# Checkpoint SHA-256 para auditoria da banca
print('\nSHA-256 dos checkpoints calibrados (para auditoria):')
for group, r in results.items():
    print(f'  Grupo {group}: {r["checkpoint_sha256"][:16]}...')

# Sanity checks
assert all(r['q_hat'] > 0 for r in results.values()), (
    'Algum q_hat saiu negativo — inspecionar antes de prosseguir para S5.8.'
)
print('\nTodos os q_hat positivos \u2014 calibracao coerente.')

CALIBRACAO CONCLUIDA — Grupos A/B/C (alpha=0.10, cobertura nominal 90%)
Grupo Metodo      q_hat         n_pixels        mean_score  
----------------------------------------------------------------------
A     ScaledCP    2.052745      70,759,312      0.997074    
B     CQR         0.010275      70,759,312      0.001729    
C     CQR         0.010269      70,759,312      0.001693    

Referencia (paper-base, Giannakopoulos et al. 2026, Tabela 1):
  Grupo B (QR), lambda_cal esperado: ~1.54
  Pearson check do Grupo B saiu em 0.978 (vs paper 0.91).
  Predicao: q_hat_B < 1.54 (replicacao com calibracao melhor).
  Predicao: q_hat_C ligeiramente menor que q_hat_B (C ja predisse
             intervalos um pouco mais largos, mean_unc 1.7% acima do B).
  q_hat_A em escala diferente (multiplicativo: u_cal = q*u(x)),
  esperado em [1.5, 2.0] para scaled CP locally-adaptive.

SHA-256 dos checkpoints calibrados (para auditoria):
  Grupo A: 9ef2fa8e4e85706e...
  Grupo B: fc185dedb3457b3c...
  Grupo 

In [16]:
# Célula 8 — Empacota e publica como Kaggle dataset.
# tcc-mri-conformal-qhats vira input do notebook S5.8 (metricas).

import shutil

KAGGLE_USER = 'massanorikishi'
DATASET_SLUG = 'tcc-mri-conformal-qhats'
DATASET_TITLE = 'TCC MRI Conformal q-hats (S5.7)'
OUTPUT_DIR = Path('/kaggle/working/tcc-mri-conformal-qhats')
OUTPUT_DIR.mkdir(exist_ok=True)

# Copia os 3 JSONs
for group in 'ABC':
    src = Path(f'/kaggle/working/q_hat_{group}.json')
    dst = OUTPUT_DIR / src.name
    shutil.copy2(src, dst)
    size_kb = src.stat().st_size / 1024
    print(f'  copiado: {src.name} ({size_kb:.1f} kB)')

# Metadata
metadata = {
    'title': DATASET_TITLE,
    'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
}
(OUTPUT_DIR / 'dataset-metadata.json').write_text(
    json.dumps(metadata, indent=2), encoding='utf-8'
)

# Cria (ou versiona se ja existir)
print('\nCriando dataset Kaggle (primeira versao)...')
result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', str(OUTPUT_DIR), '-r', 'zip'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('Create falhou, tentando version:')
    print(result.stderr)
    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(OUTPUT_DIR),
         '-m', 'Update from S5.7 notebook', '-r', 'zip'],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERRO:', result.stderr)
        raise RuntimeError('Falha ao publicar dataset.')

print(f'\nDataset URL: https://www.kaggle.com/datasets/{KAGGLE_USER}/{DATASET_SLUG}')

  copiado: q_hat_A.json (0.7 kB)
  copiado: q_hat_B.json (0.6 kB)
  copiado: q_hat_C.json (0.7 kB)

Criando dataset Kaggle (primeira versao)...
Starting upload for file q_hat_C.json
Upload successful: q_hat_C.json (673B)
Starting upload for file q_hat_A.json
Upload successful: q_hat_A.json (693B)
Starting upload for file q_hat_B.json
Upload successful: q_hat_B.json (665B)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/massanorikishi/tcc-mri-conformal-qhats


Dataset URL: https://www.kaggle.com/datasets/massanorikishi/tcc-mri-conformal-qhats


## Próximo passo: S5.8 — métricas por região + análise estatística

Com os 3 `q_hat` publicados em `tcc-mri-conformal-qhats`, o notebook `notebooks/kaggle_metrics_S5_8.ipynb` (próxima sessão) fará:

### Inputs

- 3 checkpoints (datasets já publicados)
- 3 `q_hat` (este notebook)
- Máscaras de lesão (`tcc-mri-lesion-masks`)
- Split `test` (47 volumes) do `tcc-mri-recons-varnet-brain-4x`

### Métricas a computar (sobre `test`, pós-calibração)

| Métrica | Onde | Esperado |
|---|---|---|
| **Coverage global** | Todos os pixels | ~90% para A/B/C (garantia formal) |
| **Coverage_lesion** | Pixels de lesão | C > B ≥ A (Clopper-Pearson 95% CI) |
| **Mean interval width (global)** | Todos os pixels | B/C narrower que A (idealmente) |
| **Mean interval width (lesão)** | Pixels de lesão | C ligeiramente > B (intervalos mais conservadores onde precisa) |
| **IoU_uncertainty** | Top-X% incerteza vs top-X% erro | Maior = melhor alinhamento |
| **ULAS** (contribuição original) | Gradiente alinhamento em lesões | C > B ≥ A |

### Análise estatística (S5.9)

- **Friedman test + Nemenyi post-hoc** (Demšar, 2006) para Coverage_lesion entre A/B/C
- **Holm-Bonferroni correction** para múltiplas comparações (esp. quando comparar A vs B vs C par-a-par)
- **BCa bootstrap** para CIs em subgrupos pequenos (~47 volumes test)
- **Clopper-Pearson** para Coverage_lesion (já implementado em `src/metrics/coverage.py`)

### Referências

- Romano, Y.; Patterson, E.; Candès, E. (2019). NeurIPS 32.
- Angelopoulos, A.N.; Bates, S. (2023). FnT in ML 16(4).
- Demšar, J. (2006). JMLR 7:1-30.
- Giannakopoulos, C. et al. (2026). arXiv:2601.13236, seções IV-V.